### Spectral Analysis Module(4th in operation order):

Description: Compares the number of photons received from each source at different
wavelengths (plots spectra).

### Code:

In [ ]:
#This module does the following:
#Uses variables/files from the User Input/Source Detection/Variability Modules.
#Places all newly created files in the user-specified "outputDataFolderPath"
#Gets the spectral data (.pi/.arf/.rmf files) for each source detected by the source detection module, using specextract.
#Does not take into account or subtract the background from each source.

#Import modules:

from ciao_contrib import runtool as rt
from subprocess import *
import os
from pathlib import Path
from os import listdir
from os.path import isfile, join
from lightcurves import *
from paramio import *
import paramio
from pycrates import read_file

#This is a bandaid fix needed to clear any previous plots and fix some issues with matplotlib. Remove if matplotlib isn’t causing any issues.
%matplotlib inline 
import matplotlib.pyplot as plt
plt.clf()

from sherpa.astro.ui import *
from sherpa_contrib.notebook_plotter import notebook_plotter

#Define functions:

#None

#Set variables:

specextractAsol1File = variabilityInputAsol1File
specextractBpix1File = variabilityInputBpix1File
specextractMsk1File = variabilityInputMsk1File

#How spectra are checked for each source:
#Loop through each source region in the list of regions, "regionList", made by the Source Detection Module,
#Run specextract to get the spectra for each source,
#Then plot the data:

print("Checking spectra for", len(regionList), "sources:\n")
i = 1
while i <= len(regionList):
    
    #A background region file (containing the area surrounding each source) is optional for specextract, so it isn't included here, but it should be included so that the background can be subtracted from each source.
    #The CIAO threads related to specextract manually create this background region for each source, but an automatic method for doing this is not as straightforward.

    specextractRegion = regionList[i-1]
    specextractInfile = variabilityInputEvt2File+"[sky="+specextractRegion+"]"
    specextractOutroot = outputDataFolderPath+"/specextract_region_"+str(i)
    print("\nCurrent source region (", i, ")\n", specextractRegion)
    
    #Setting the rest of the parameters for specextract (the asp, mskfile, and badpixfile parameters need to be specified to prevent an error if the input path and output path are not in the same location, and the three respective files can't be found)
    
    pset ("specextract", "asp", specextractAsol1File)
    pset ("specextract", "mskfile", specextractMsk1File)
    pset ("specextract", "badpixfile", specextractBpix1File)
    pset ("specextract", "clobber", "yes")

    #Run specextract:
    
    call(["specextract", specextractInfile, specextractOutroot, "correctpsf=yes", "weight=no"])
    
    i+=1
